#NOTEBOOK COLETA E LIMPEZA DE DADOS

O Dataset utilizado neste estudo foi fornecido pela FIAP. Trata-se de um arquivo CSV com dados de pesquisa sobre hábitos da população, direcionados a incidência de obesidade.

O arquivo contém alguma colunas com "ruídos", dados que precisam ser identificados, comparados aos limites do dicionário de dados fornecido e tratados. A limpeza e tratamento de dados precisa garantir qualidade e integridade.A limpeza de dados é fundamental para garantir a precisão e confiabilidade dos resultados da análise, pois dados sujos ou inconsistentes podem levar a conclusões equivocadas ou imprecisas.

In [33]:
import pandas as pd
import numpy as np

##IMPORTACAO DO ARQUIVO ORIGINAL QUE ESTÁ NO GITHUB

In [34]:
# ==========================================
# IMPORTAÇÃO
# ==========================================
caminho_ficheiro = 'https://raw.githubusercontent.com/LuciAguiar/Tech_Chalenge_Obesity/refs/heads/main/Obesity.csv'
df_obesity = pd.read_csv(caminho_ficheiro, sep=',', decimal=',', encoding='latin1')

In [35]:
df_obesity.head()

,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21,1.62,64,yes,no,2,3,Sometimes,no,2,no,0,1,no,Public_Transportation,Normal_Weight
1,Female,21,1.52,56,yes,no,3,3,Sometimes,yes,3,yes,3,0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23,1.8,77,yes,no,2,3,Sometimes,no,2,no,2,1,Frequently,Public_Transportation,Normal_Weight
3,Male,27,1.8,87,no,no,3,3,Sometimes,no,2,no,2,0,Frequently,Walking,Overweight_Level_I
4,Male,22,1.78,89.8,no,no,2,1,Sometimes,no,2,no,0,0,Sometimes,Public_Transportation,Overweight_Level_II


##LIMPEZA E VERIFICAÇÃO DOS DADOS

Nesta etapa também é verificado se os dados estão dentro dos limites estabelecidos no dicionário de dados

In [36]:
# ==========================================
# TRATAMENTO DA COLUNA 'AGE' (Idade)
# ==========================================
def limpar_age_vectorized(age_series, limite_inf, limite_sup):
    s_original = age_series.astype(str).str.strip()
    cleaned_values = pd.Series(np.nan, index=age_series.index, dtype='float64')

    # conversão numérica (e.g., '21.' -> 21, '21,0' -> 21)
    s_cleaned_for_numeric = s_original.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    num_robust = pd.to_numeric(s_cleaned_for_numeric, errors='coerce')

    # Remove ponto e pega os 2 primeiros dígitos
    s_original_no_dot = s_original.str.replace('.', '', regex=False)
    first_two_digits_str = s_original_no_dot.str[:2]
    is_first_two_digits_numeric = first_two_digits_str.str.isdigit()
    first_two_digits_num = pd.to_numeric(first_two_digits_str, errors='coerce')

    # Verifica se o valor convertido está entre os limites inferiores e superiores definidos no dicionário de dados
    condition_robust_in_range = (num_robust >= limite_inf) & (num_robust <= limite_sup)
    cleaned_values.loc[condition_robust_in_range] = num_robust.loc[condition_robust_in_range].round()

    # Verifica se houve falha na conversão
    condition_fallback_applies = cleaned_values.isna() & \
                                 is_first_two_digits_numeric & \
                                 (first_two_digits_num >= limite_inf) & (first_two_digits_num <= limite_sup)

    cleaned_values.loc[condition_fallback_applies] = first_two_digits_num.loc[condition_fallback_applies]

    return cleaned_values.astype('Int64')

df_obesity['Age'] = limpar_age_vectorized(df_obesity['Age'], 14, 61)

# ==========================================
# TRATAMENTO DA COLUNA 'HEIGHT' (Altura)
# ==========================================
def padronizar_altura(valor):
    if pd.isna(valor):
        return np.nan
    v_str = str(valor).strip()
    v_str_normalized = v_str.replace(',', '.')

    final_height_num = np.nan

    try:
        altura_num = float(v_str_normalized)
        if 1.45 <= altura_num <= 1.98:
            final_height_num = altura_num
    except ValueError:
        pass

    # Compara com os limites e faz a formatação
    if pd.isna(final_height_num):
        v_digits = ''.join(filter(str.isdigit, v_str)) # Extract only digits
        if len(v_digits) >= 3:
            # Form X.YY from first 3 digits
            formed_height_str = f"{v_digits[0]}.{v_digits[1:3]}"
            try:
                altura_num_formed = float(formed_height_str)
                if 1.45 <= altura_num_formed <= 1.98:
                    final_height_num = altura_num_formed
            except ValueError:
                pass

    # se a lógica anterior não funcionou, tenta fazer a conversão de outra forma
    if not pd.isna(final_height_num):
        return int(final_height_num * 100) / 100.0
    else:
        return np.nan #

df_obesity['Height'] = df_obesity['Height'].apply(padronizar_altura)

# ==========================================
# TRATAMENTO DA COLUNA 'WEIGHT' (Peso)
# ==========================================
def padronizar_peso(valor):
    if pd.isna(valor): return np.nan
    v = str(valor).strip().replace('.', '')
    while len(v) > 0:
        try:
            peso_num = float(v)
            if 39 <= peso_num <= 173: return peso_num
        except ValueError:
            pass
        v = v[:-1]
    return np.nan

df_obesity['Weight'] = df_obesity['Weight'].apply(padronizar_peso)

# =============================================================
# TRATAMENTO DAS CATEGORIAS COM RUÍDO COM RESULTADO NUMERICO
# =============================================================
# Função auxiliar genérica para colunas categóricas com limites numéricos
def limpar_categorica(coluna_nome, limite_inf, limite_sup):
    num = pd.to_numeric(df_obesity[coluna_nome], errors='coerce').round()
    condicao = (num >= limite_inf) & (num <= limite_sup)
    return np.select([condicao], [num], default=np.nan)

def limpar_coluna_first_digit_fallback(series, limite_inf, limite_sup):
    s_original = series.astype(str).str.strip()
    cleaned_values = pd.Series(np.nan, index=series.index, dtype='float64')

    s_cleaned_for_numeric = s_original.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    num_robust = pd.to_numeric(s_cleaned_for_numeric, errors='coerce')

    first_digit_str = s_original.str[0]
    is_first_digit_numeric = first_digit_str.str.isdigit()
    first_digit_num = pd.to_numeric(first_digit_str, errors='coerce')

    # Prioriza conversão numéri a se estiver dentro dos limites
    condition_robust_in_range = (num_robust >= limite_inf) & (num_robust <= limite_sup)
    cleaned_values.loc[condition_robust_in_range] = num_robust.loc[condition_robust_in_range].round()

    # Compara com os limites inferiores e superiores e tenta outra estratégia em casa de falha
    condition_fallback_applies = cleaned_values.isna() & \
                                 is_first_digit_numeric & \
                                 (first_digit_num >= limite_inf) & (first_digit_num <= limite_sup)

    cleaned_values.loc[condition_fallback_applies] = first_digit_num.loc[condition_fallback_applies]

    return cleaned_values.astype('Int64')

# Aplicando as funções de limpeza às colunas
df_obesity['FCVC'] = limpar_coluna_first_digit_fallback(df_obesity['FCVC'], 1, 3)
df_obesity['NCP']  = limpar_coluna_first_digit_fallback(df_obesity['NCP'], 1, 4)
df_obesity['CH2O'] = limpar_coluna_first_digit_fallback(df_obesity['CH2O'], 1, 3)
df_obesity['FAF']  = limpar_coluna_first_digit_fallback(df_obesity['FAF'], 0, 3)
df_obesity['TUE']  = limpar_coluna_first_digit_fallback(df_obesity['TUE'], 0, 2)

# ==========================================
# CONVERSÃO FINAL PARA INTEIROS
# ==========================================
# Lista de todas as colunas que devem ser números inteiros puros
colunas_inteiras = ['Age', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

# Converte usando o tipo especial 'Int64' (que suporta valores nulos mantendo-se inteiro)
for coluna in colunas_inteiras:
    df_obesity[coluna] = df_obesity[coluna].astype('Int64')

##VERIFICAÇÃO DAS COLUNAS TRATADAS

In [37]:
# ==========================================
# 7. VERIFICAÇÃO FINAL
# ==========================================
# visualização das colunas tratadas
colunas_limpas = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

print("Visualização das primeiras 10 linhas")
display(df_obesity[colunas_limpas].head(10))

print("\nTipos de dados atualizados:")
print(df_obesity[colunas_limpas].dtypes)

Visualização das primeiras 10 linhas


,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE
0,21,1.62,64,2,3,2,0,1
1,21,1.52,56,3,3,3,3,0
2,23,1.80,77,2,3,2,2,1
3,27,1.80,87,3,3,2,2,0
4,22,1.78,89,2,1,2,0,0
5,29,1.62,53,2,3,2,0,0
6,23,1.50,55,3,3,2,1,0
7,22,1.64,53,2,3,2,3,0
8,24,1.78,64,3,3,2,1,1
9,22,1.72,68,2,3,2,1,1



Tipos de dados atualizados:
Age         Int64
Height    float64
Weight      Int64
FCVC        Int64
NCP         Int64
CH2O        Int64
FAF         Int64
TUE         Int64
dtype: object


##CONFERENCIA SE AINDA FICOU ALGUM VALOR NULO

In [38]:
print("Visualizing all rows with any null values:")
rows_with_null = df_obesity[df_obesity.isnull().any(axis=1)]
display(rows_with_null.head()) # Displaying the first few rows with nulls, as there might be many

print(f"\nTotal number of rows with at least one null value: {len(rows_with_null)}")

Visualizing all rows with any null values:


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity



Total number of rows with at least one null value: 0


##EXPORTACAO DO ARQUIVO COM OS DADOS TRATADOS PARA SEREM USADOS NA PROXIMA ETAPA - ANÁLISE EXPLORATÓRIA DE DADOS


In [39]:
df_obesity.to_excel('Obesity_Limpo.xlsx', index=False)
df_obesity.to_csv('Obesity_Limpo.csv', index=False)
